In [1]:
# Imports
import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model





In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

In [3]:

BASE_DIR = "../dataset" 
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

print(f"Train path exist: {os.path.exists(TRAIN_DIR)}")
print(f"Test path exist: {os.path.exists(TEST_DIR)}")

Train path exist: True
Test path exist: True


In [4]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
    )

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [5]:

X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 


Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


In [6]:


num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 134s 1s/step - accuracy: 0.4024 - loss: 1.2849 - val_accuracy: 0.2437 - val_loss: 2.0220
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 74s 818ms/step - accuracy: 0.5495 - loss: 1.0555 - val_accuracy: 0.3401 - val_loss: 1.9042
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 73s 805ms/step - accuracy: 0.6000 - loss: 0.9017 - val_accuracy: 0.3503 - val_loss: 2.3768
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 73s 804ms/step - accuracy: 0.6202 - loss: 0.8808 - val_accuracy: 0.3629 - val_loss: 2.5150
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 72s 793ms/step - accuracy: 0.6474 - loss: 0.8058 - val_accuracy: 0.3655 - val_loss: 2.6867
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 72s 802ms/step - accuracy: 0.6582 - loss: 0.7814 - val_accuracy: 0.3655 - val_loss: 2.9880
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 71s 788ms/step - accuracy: 0.6840 - loss: 0.7516 - val_accuracy: 0.4086 - val_loss: 3.0252
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 72s 794ms/step - accuracy: 0.6777 - loss: 0.7565 - val_accura

In [9]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 142ms/step - accuracy: 0.4315 - loss: 2.7771
Baseline Loss: 2.7771339416503906
Baseline Accuracy: 0.4314720928668976


In [10]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 150ms/step
Confusion Matrix:
 [[13 15 66  6]
 [ 2 34 73  6]
 [ 3  6 96  0]
 [ 3  5 39 27]]

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.62      0.13      0.21       100
meningioma_tumor       0.57      0.30      0.39       115
        no_tumor       0.35      0.91      0.51       105
 pituitary_tumor       0.69      0.36      0.48        74

        accuracy                           0.43       394
       macro avg       0.56      0.43      0.40       394
    weighted avg       0.55      0.43      0.39       394



In [11]:

os.makedirs("../models", exist_ok=True)

model_baseline.save("../models/baseline_cnn.keras")
print("Saved to ../models/baseline_cnn.keras")

Saved to ../models/baseline_cnn.keras


# Transfer Learning Generators

In [12]:
transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


# Transfer Learning Model

In [13]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(128, activation='relu')(X)
X=Dropout(0.5)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

# Compile

In [14]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
EPOCHS_TL = 10
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.6784 - loss: 0.8197 - val_accuracy: 0.5711 - val_loss: 1.1308
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - accuracy: 0.7913 - loss: 0.5413 - val_accuracy: 0.5939 - val_loss: 1.1927
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - accuracy: 0.8209 - loss: 0.4467 - val_accuracy: 0.5812 - val_loss: 1.3381
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 81s 890ms/step - accuracy: 0.8275 - loss: 0.4184 - val_accuracy: 0.6447 - val_loss: 1.4300
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 79s 877ms/step - accuracy: 0.8411 - loss: 0.4032 - val_accuracy: 0.6650 - val_loss: 1.3435
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 82s 912ms/step - accuracy: 0.8557 - loss: 0.3619 - val_accuracy: 0.6142 - val_loss: 1.4248
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 79s 872ms/step - accuracy: 0.8686 - loss: 0.3320 - val_accuracy: 0.6827 - val_loss: 1.3449
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 81s 894ms/step - accuracy: 0.8610 - loss: 0.3476 - val_accuracy: 

In [16]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 406ms/step - accuracy: 0.6777 - loss: 1.3882
Transfer Learning Loss: 1.3881510496139526


In [17]:
print('Comparision:')
print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

Comparision:
Baseline Accuracy: 0.4314720928668976
Transfer Learning Accuracy: 0.6776649951934814


In [18]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=5
)

Epoch 1/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - accuracy: 0.6551 - loss: 0.9768 - val_accuracy: 0.6548 - val_loss: 1.8286
Epoch 2/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 90s 996ms/step - accuracy: 0.8125 - loss: 0.5013 - val_accuracy: 0.6320 - val_loss: 2.0584
Epoch 3/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 85s 941ms/step - accuracy: 0.8383 - loss: 0.4118 - val_accuracy: 0.6269 - val_loss: 2.1180
Epoch 4/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 82s 908ms/step - accuracy: 0.8582 - loss: 0.3673 - val_accuracy: 0.6523 - val_loss: 1.9575
Epoch 5/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 83s 924ms/step - accuracy: 0.8714 - loss: 0.3198 - val_accuracy: 0.6675 - val_loss: 1.9752


In [19]:
import os
os.makedirs("../models", exist_ok=True)
model_transfer.save("../models/mobilenetv2_transfer.keras")
print("Saved to ../models/mobilenetv2_transfer.keras")


Saved to ../models/mobilenetv2_transfer.keras
